In [ ]:
import numpy as np
import pandas as pd
from rdkit import Chem
import torch
from torch_geometric.data import Data
from sklearn.model_selection import train_test_split

#### Loads data from files generated in data.ipynb

In [ ]:
df_main = pd.read_csv("data_mingap.csv")
df_training = pd.read_csv("data/train.csv")
df_evaluation = pd.read_csv("data/evaluate.csv")

#set current df to training df
df = df_training

stats = {
    "mean": df_training["target_raw"].mean(),
    "std": df_training["target_raw"].std()
}

#### Adds smiles strings to laoded dataframes and normalizes (again) using the Z score

In [ ]:
def id_to_smiles(id_str, is_diene=True):
    # Mapping based on the lookup table provided
    lookup = {
        '1': 'F',
        '2': 'C#N',
        '3': 'OC',
        '4': 'C',
        '5': 'C(C)(C)C',
        '6': '', # Hydrogen (implicit)
        '7': 'c1ccccc1',
        '8': 'C(=O)OC',
        '9': 'C=O'
    }
    
    parts = str(id_str).split('_')
    groups = [lookup.get(p, '') for p in parts]

    if is_diene:
        # Scaffold: C1=C2-C3=C4
        # We attach groups to the carbons. 
        # RDKit SMILES format: [G1]C=C([G2])C([G3])=C[G4]
        g1, g2, g3, g4 = groups
        
        # Build pieces; handle empty strings (hydrogens) gracefully
        c1 = f"({g1})" if g1 else ""
        c2 = f"({g2})" if g2 else ""
        c3 = f"({g3})" if g3 else ""
        c4 = f"({g4})" if g4 else ""
        
        smiles = f"C{c1}=C{c2}C{c3}=C{c4}"
    else:
        # Scaffold: C1=C2
        # RDKit format: [G1]C=C[G2]
        g1, g2 = groups
        c1 = f"({g1})" if g1 else ""
        c2 = f"({g2})" if g2 else ""
        smiles = f"C{c1}=C{c2}"

    # Canonicalize to ensure the SMILES is clean and valid
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        return Chem.MolToSmiles(mol)
    return smiles # Return raw if RDKit fails

#adds new columns with smiles strings to both training/evaluation dataframes
df['diene_smiles'] = df['diene'].apply(lambda x: id_to_smiles(x, is_diene=True))
df['dienophile_smiles'] = df['dienophile'].apply(lambda x: id_to_smiles(x, is_diene=False))
df_evaluation['diene_smiles'] = df_evaluation['diene'].apply(lambda x: id_to_smiles(x, is_diene=True))
df_evaluation['dienophile_smiles'] = df_evaluation['dienophile'].apply(lambda x: id_to_smiles(x, is_diene=False))

#standardize with z-score for min_ts_all
ts_mean = df["Min_TSـall"].mean()
ts_std = df["Min_TSـall"].std()
df["Min_TS_scaled"] =  (df["Min_TSـall"] - ts_mean) / ts_std
df_evaluation["Min_TS_scaled"] =  (df_evaluation["Min_TSـall"] - ts_mean) / ts_std

#### Creates datasets for both training/evaluation of pyg objects

In [ ]:
def mol_to_pyg_data(smiles, pz_pops, nbo_charges, volumes):
    """
    Converts a molecule and its CSV electronic data into a PyG Data object.
    """
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol) # Add hydrogens for accurate geometry/physics
    frags = Chem.GetMolFrags(mol, asMols=True)
    
    # 1. Node Features (x)
    node_features = []
    for k in range(len(frags)):
        for i, atom in enumerate(frags[k].GetAtoms()):
            d_or_dph = float(0.0)

            # Atomic Number (e.g., 6 for Carbon)
            atomic_num = atom.GetAtomicNum()
            
            # Pull electronic data from CSV (defaulting to 0 if not a core C1-C4 atom)
            pz = pz_pops[i] if i < len(pz_pops) else 0.0
            nbo = nbo_charges[i] if i < len(nbo_charges) else 0.0
            vol = volumes[i] if i < len(volumes) else 0.0
            
            node_features.append([atomic_num, pz, nbo, vol, 0.0 if k == 0 else 1.0]) # Last feature indicates fragment (diene/dienophile)
    
    x = torch.tensor(node_features, dtype=torch.float)

    # 2. Edge Index (Adjacency)
    edges = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edges.append([i, j])
        edges.append([j, i]) # Undirected graph
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    return Data(x=x, edge_index=edge_index)

def create_reaction_graph_from_row(row):
    diene_smi = id_to_smiles(row['diene'], is_diene=True)
    dienophile_smi = id_to_smiles(row['dienophile'], is_diene=False)
    combined_smi = f"{diene_smi}.{dienophile_smi}"
    
    # 2. Combine Core Carbon features using your actual headers: dPh
    pz_list = [row['pz_pop_C1_D'], row['pz_pop_C2_D'], row['pz_pop_C3_D'], row['pz_pop_C4_D'], 
               row['pz_pop_C1_dPh'], row['pz_pop_C2_dPh']]
    
    # Based on headers: NBO_1_D... and NBO_1dPh / NBO_2_dPh
    nbo_list = [row['NBO_1_D'], row['NBO_2_D'], row['NBO_3_D'], row['NBO_4_D'], 
                row['NBO_1dPh'], row['NBO_2_dPh']]
    
    # Based on headers: Volume_D_1... and Volume_dPh_1...
    vol_list = [row['Volume_D_1'], row['Volume_D_2'], row['Volume_D_3'], row['Volume_D_4'], 
                row['Volume_dPh_1'], row['Volume_dPh_2']]
    
    graph = mol_to_pyg_data(combined_smi, pz_list, nbo_list, vol_list)
    graph.y = torch.tensor([row['Min_TS_scaled']], dtype=torch.float)
    return graph

dataset = [create_reaction_graph_from_row(row) for _, row in df.iterrows()] 
evaluation_dataset = [create_reaction_graph_from_row(row) for _, row in df_evaluation.iterrows()]

print(f"Standardized Mean: {df['Min_TS_scaled'].mean():.2f}") # Should be 0.0
print(f"Standardized Std:  {df['Min_TS_scaled'].std():.2f}")  # Should be 1.0

#test
create_reaction_graph_from_row(df.iloc[0])

#### Transformer Class

In [ ]:
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU, BatchNorm1d
from torch_geometric.nn import TransformerConv, global_mean_pool

class DielsAlderTransformer(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim=64, heads=4):
        super(DielsAlderTransformer, self).__init__()
        
        global_dropout = 0.1

        self.conv1 = TransformerConv(input_dim, hidden_dim, heads=heads, dropout=global_dropout)
        self.bn1 = BatchNorm1d(hidden_dim * heads)
        
        self.conv2 = TransformerConv(hidden_dim * heads, hidden_dim, heads=heads, dropout=global_dropout)
        self.bn2 = BatchNorm1d(hidden_dim * heads)

        self.conv3 = TransformerConv(hidden_dim * heads, hidden_dim, heads=heads, dropout=global_dropout)
        self.bn3 = BatchNorm1d(hidden_dim * heads)

        # Output MLP
        self.mlp = Sequential(
            Linear(hidden_dim * heads, hidden_dim),
            ReLU(),
            Linear(hidden_dim, 1)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # Layer 1
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        
        # Layer 2
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        
        # Layer 3
        x = F.relu(self.bn3(self.conv3(x, edge_index)))

        # Global Pooling now averages over BOTH diene and dienophile atoms
        x = global_mean_pool(x, batch)
        
        return self.mlp(x)

#### Model Training

In [ ]:
from unittest import loader
from torch_geometric.loader import DataLoader

def train_model(model, train_loader, optimizer, device):
    model.train()
    total_loss = 0
    
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        
        # Forward pass
        prediction = model(data)
        
        # target (y) should be (batch_size, 1)
        loss = F.mse_loss(prediction.view(-1), data.y.view(-1))
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * data.num_graphs
        
    return total_loss / len(train_loader.dataset)


print(torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
model = DielsAlderTransformer(input_dim=5).to(device)

adam_optimizer = torch.optim.Adam(model.parameters(), lr=0.005) # ~44 loss pre min_ts_all 
#0.38 final loss with adam @ 0.005 lr
adamW_optimizer = torch.optim.AdamW(model.parameters(), lr=0.005)
#dont use sgd
sgd_optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

loader = DataLoader(dataset, batch_size=32, shuffle=True)

for epoch in range(200):
   loss = train_model(model, loader, adam_optimizer, device)
   print(f"Epoch {epoch:03d}, Loss: {loss:.6f}")

#### Model Evaluation

In [ ]:
import pandas as pd
import torch
from torch_geometric.loader import DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_on_test(model, test_loader, ts_mean, ts_std):
    model.eval()
    all_preds = []
    all_actuals = []
    
    with torch.no_grad():
        for data in test_loader:
            data = data.to(device)
            out = model(data)
            # Unscale the predictions and targets back to kcal/mol
            pred_kcal = (out.cpu().numpy().flatten() * ts_std) + ts_mean
            actual_kcal = (data.y.cpu().numpy().flatten() * ts_std) + ts_mean
            
            all_preds.extend(pred_kcal)
            all_actuals.extend(actual_kcal)

    mae = mean_absolute_error(all_actuals, all_preds)
    rmse = np.sqrt(mean_squared_error(all_actuals, all_preds))
    r2 = r2_score(all_actuals, all_preds)

    print(f"Test MAE:  {mae:.2f} kcal/mol")
    print(f"Test RMSE: {rmse:.2f} kcal/mol")
    print(f"Test R2:   {r2:.4f}")
    
    return all_actuals, all_preds

def evaluate_external_dataset(model, ts_mean, ts_std):
    # 1. Load the new 200-reaction dataset
    df_new = df_evaluation
    
    # 2. Convert to PyG Graphs (Using your existing graph creation function)
    # Important: Apply the OLD ts_mean and ts_std for scaling
    new_graphs = []
    for _, row in df_new.iterrows():
        # Use your previous function here, e.g., create_graph_from_smiles(row['smiles'])
        graph = create_reaction_graph_from_row(row) 
        
        # Scale the new targets using the original training stats
        scaled_y = (row['Min_TSـall'] - ts_mean) / ts_std
        graph.y = torch.tensor([scaled_y], dtype=torch.float)
        new_graphs.append(graph)
    
    # 3. Create Loader
    external_loader = DataLoader(new_graphs, batch_size=32, shuffle=False)
    
    # 4. Run Evaluation
    print(f"--- Evaluating External Dataset: {df_new} ---")
    actuals, predictions = evaluate_on_test(model, external_loader, ts_mean, ts_std)
    
    return actuals, predictions

# Execute
actuals_200, preds_200 = evaluate_external_dataset(model, stats['mean'], stats['std'])